In [2]:
#!/usr/bin/env python
# coding: utf-8

"""
🏪 Store People Counter - ENHANCED ZONE
- منطقة مستطيلة تغطي باب المحل
- أي جسم (جزء من الجسم أو لباس) يلمس المنطقة يتم عده
- تجاهل الأجسام الصغيرة جداً (ذباب، حشرات، غبار)
- كل شخص يُحسب مرة واحدة فقط
- دقة عالية في العد
"""

import cv2
import numpy as np
from ultralytics import YOLO
import time
import os
import threading
from collections import deque
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
import socket
import sys
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. GET IP ADDRESS
# ============================================================

def get_local_ip():
    try:
        hostname = socket.gethostname()
        ip_address = socket.gethostbyname(hostname)
        return ip_address
    except:
        return "127.0.0.1"

CAMERA_IP = get_local_ip()
print(f"📡 Camera IP: {CAMERA_IP}")

# ============================================================
# 2. EXCEL LOGGING SYSTEM
# ============================================================

try:
    import pandas as pd
    from datetime import datetime
    import openpyxl
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
    EXCEL_AVAILABLE = True
    print("📊 Excel logging enabled")
except:
    EXCEL_AVAILABLE = False
    print("⚠️ Excel not available")

class ExcelLogger:
    def __init__(self, filename="E:\PPE-Detect-system\PPE-Detect-system\store_log.xlsx"):
        self.filename = filename
        self.data = []
        self.headers = ['IP_Address', 'Date', 'Time', 'Entered_Count', 'Exited_Count', 'Inside_Count']
        self.camera_ip = CAMERA_IP
        self.batch_data = []
        
        if not os.path.exists(filename):
            self.create_new_file()
    
    def create_new_file(self):
        try:
            df = pd.DataFrame(columns=self.headers)
            df.to_excel(self.filename, index=False, engine='openpyxl')
            print(f"📊 Created new log file: {self.filename}")
        except Exception as e:
            print(f"⚠️ Error creating file: {e}")
    
    def add_entry(self, entered_count, exited_count, inside_count):
        now = datetime.now()
        entry = {
            'IP_Address': self.camera_ip,
            'Date': now.strftime("%Y-%m-%d"),
            'Time': now.strftime("%H:%M:%S"),
            'Entered_Count': entered_count,
            'Exited_Count': exited_count,
            'Inside_Count': inside_count
        }
        
        self.data.append(entry)
        self.batch_data.append(entry)
        
        if len(self.batch_data) >= 5:
            self.save_batch()
    
    def save_batch(self):
        if not self.batch_data:
            return
        try:
            df_batch = pd.DataFrame(self.batch_data)
            if os.path.exists(self.filename):
                existing_df = pd.read_excel(self.filename, engine='openpyxl')
                updated_df = pd.concat([existing_df, df_batch], ignore_index=True)
                updated_df.to_excel(self.filename, index=False, engine='openpyxl')
            else:
                df_batch.to_excel(self.filename, index=False, engine='openpyxl')
            self.batch_data = []
        except Exception as e:
            print(f"⚠️ Excel save error: {e}")
    
    def save_final(self):
        if self.batch_data:
            self.save_batch()
        print(f"📊 Final data saved to: {self.filename}")

# ============================================================
# 3. SOUND SYSTEM
# ============================================================

SOUND_FILE = r"beep-01a.wav"

try:
    import winsound
    SOUND_ENABLED = True
except:
    try:
        from playsound import playsound
        SOUND_ENABLED = True
    except:
        SOUND_ENABLED = False

def play_sound():
    if not SOUND_ENABLED:
        return
    try:
        if os.path.exists(SOUND_FILE):
            try:
                winsound.PlaySound(SOUND_FILE, winsound.SND_FILENAME | winsound.SND_ASYNC)
            except:
                try:
                    from playsound import playsound
                    playsound(SOUND_FILE, block=False)
                except:
                    pass
        else:
            try:
                winsound.Beep(1000, 200)
            except:
                pass
    except:
        pass

# ============================================================
# 4. TRACKER - ENHANCED
# ============================================================

class Track(object):
    def __init__(self, detection, track_id):
        self.track_id = track_id
        self.kf = KalmanFilter(dim_x=8, dim_z=4)
        
        self.kf.F = np.array([[1, 0, 0, 0, 1, 0, 0, 0],
                              [0, 1, 0, 0, 0, 1, 0, 0],
                              [0, 0, 1, 0, 0, 0, 1, 0],
                              [0, 0, 0, 1, 0, 0, 0, 1],
                              [0, 0, 0, 0, 1, 0, 0, 0],
                              [0, 0, 0, 0, 0, 1, 0, 0],
                              [0, 0, 0, 0, 0, 0, 1, 0],
                              [0, 0, 0, 0, 0, 0, 0, 1]])
        
        self.kf.H = np.array([[1, 0, 0, 0, 0, 0, 0, 0],
                              [0, 1, 0, 0, 0, 0, 0, 0],
                              [0, 0, 1, 0, 0, 0, 0, 0],
                              [0, 0, 0, 1, 0, 0, 0, 0]])
        
        self.kf.R[2:, 2:] *= 3.0
        self.kf.P[4:, 4:] *= 1000.0
        self.kf.P *= 10.0
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01
        
        detection_array = np.array(detection[:4]).reshape(4, 1)
        self.kf.x[:4] = detection_array
        self.kf.x[4:] = 0
        
        self.hits = 0
        self.no_losses = 0
        self.time_since_update = 0
        
        self.positions = deque(maxlen=30)
        self.positions.append((detection[0], detection[1]))
        
        self.bbox_history = deque(maxlen=30)
        self.bbox_history.append((detection[0], detection[1], detection[2], detection[3]))
        
        self.counted = False
        self.count_direction = None
        self.movement_speed = 0.0
        self.direction_confidence = 0.0
        self.smooth_y = detection[1]
        self.smooth_x = detection[0]
        
        self.detection_width = detection[2]
        self.detection_height = detection[3]
        
        self.movement_pattern = deque(maxlen=15)
        self.movement_pattern.append(detection[1])
        
        self.zone_history = deque(maxlen=10)
        self.zone_history.append(False)
        
        self.cross_attempts = 0
        self.confirmed_cross = False
        self.cross_direction = None
        self.counted_once = False
        
        # تحديد إذا كان الجسم صغير جداً (ذباب، حشرات)
        self.is_small_object = False
        self.update_small_object_status()
    
    def update_small_object_status(self):
        """تحديد إذا كان الجسم صغيراً جداً (أقل من الحد الأدنى)"""
        min_width = 25
        min_height = 25
        if self.detection_width < min_width or self.detection_height < min_height:
            self.is_small_object = True
        else:
            self.is_small_object = False
    
    def update(self, detection):
        detection_array = np.array(detection[:4]).reshape(4, 1)
        self.kf.update(detection_array)
        self.hits += 1
        self.no_losses = 0
        self.time_since_update = 0
        
        self.positions.append((detection[0], detection[1]))
        self.bbox_history.append((detection[0], detection[1], detection[2], detection[3]))
        self.smooth_y = 0.7 * self.smooth_y + 0.3 * detection[1]
        self.smooth_x = 0.7 * self.smooth_x + 0.3 * detection[0]
        self.movement_pattern.append(detection[1])
        
        self.detection_width = detection[2]
        self.detection_height = detection[3]
        self.update_small_object_status()
        
        if len(self.positions) >= 2:
            p1 = self.positions[-2]
            p2 = self.positions[-1]
            self.movement_speed = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
    
    def predict(self):
        self.kf.predict()
        self.time_since_update += 1
    
    def get_state(self):
        return self.kf.x[:4].flatten()
    
    def get_center(self):
        state = self.get_state()
        return (state[0], state[1])
    
    def get_bbox(self):
        state = self.get_state()
        return (state[0], state[1], state[2], state[3])
    
    def get_movement_speed(self):
        return self.movement_speed
    
    def get_size(self):
        return self.detection_width, self.detection_height
    
    def is_valid_object(self):
        """التحقق من أن الجسم ليس صغيراً جداً (تجاهل الذباب والحشرات)"""
        if self.is_small_object:
            return False
        return True
    
    def is_child_person(self, min_width=70, min_height=130):
        if self.detection_width < min_width or self.detection_height < min_height:
            return True
        return False
    
    def get_movement_trend(self):
        if len(self.movement_pattern) < 5:
            return 0
        pattern = list(self.movement_pattern)
        x = np.arange(len(pattern))
        y = np.array(pattern)
        slope = np.polyfit(x, y, 1)[0]
        return slope
    
    def check_zone_intersection(self, zone_x1, zone_y1, zone_x2, zone_y2):
        """التحقق من ملامسة أي جزء من الجسم للمنطقة"""
        if len(self.bbox_history) < 1:
            return False
        
        cx, cy, w, h = self.bbox_history[-1]
        
        # حساب أطراف المربع المحيط
        bbox_x1 = cx - w/2
        bbox_y1 = cy - h/2
        bbox_x2 = cx + w/2
        bbox_y2 = cy + h/2
        
        # AABB Collision Detection
        if (bbox_x1 < zone_x2 and bbox_x2 > zone_x1 and
            bbox_y1 < zone_y2 and bbox_y2 > zone_y1):
            return True
        
        return False
    
    def check_zone_crossing(self, zone_x1, zone_y1, zone_x2, zone_y2):
        """التحقق من دخول أو خروج الجسم من المنطقة"""
        if len(self.bbox_history) < 3:
            return None, False
        
        # الحصول على المربعات الحالية والسابقة
        current_cx, current_cy, current_w, current_h = self.bbox_history[-1]
        prev_cx, prev_cy, prev_w, prev_h = self.bbox_history[-2]
        
        # حساب المربع الحالي
        current_bbox_x1 = current_cx - current_w/2
        current_bbox_y1 = current_cy - current_h/2
        current_bbox_x2 = current_cx + current_w/2
        current_bbox_y2 = current_cy + current_h/2
        
        # حساب المربع السابق
        prev_bbox_x1 = prev_cx - prev_w/2
        prev_bbox_y1 = prev_cy - prev_h/2
        prev_bbox_x2 = prev_cx + prev_w/2
        prev_bbox_y2 = prev_cy + prev_h/2
        
        # التحقق من وجود الجسم داخل المنطقة حالياً
        current_inside = (current_bbox_x1 < zone_x2 and current_bbox_x2 > zone_x1 and
                         current_bbox_y1 < zone_y2 and current_bbox_y2 > zone_y1)
        
        # التحقق من وجود الجسم داخل المنطقة سابقاً
        prev_inside = (prev_bbox_x1 < zone_x2 and prev_bbox_x2 > zone_x1 and
                      prev_bbox_y1 < zone_y2 and prev_bbox_y2 > zone_y1)
        
        self.zone_history.append(current_inside)
        
        crossed = False
        direction = None
        confidence = 0.0
        
        # ============================================
        # الطريقة 1: الدخول أو الخروج المباشر
        # ============================================
        if not prev_inside and current_inside:
            crossed = True
            direction = "IN"
            confidence = 0.98
        
        elif prev_inside and not current_inside:
            crossed = True
            direction = "OUT"
            confidence = 0.98
        
        # ============================================
        # الطريقة 2: الحركة داخل المنطقة
        # ============================================
        if not crossed and current_inside and len(self.zone_history) >= 5:
            inside_count = sum(1 for x in self.zone_history if x)
            if inside_count >= 3:
                trend = self.get_movement_trend()
                speed = self.get_movement_speed()
                
                if speed > 2.0:
                    if trend < -1.0:
                        crossed = True
                        direction = "OUT"
                        confidence = 0.75
                    elif trend > 1.0:
                        crossed = True
                        direction = "IN"
                        confidence = 0.75
        
        # ============================================
        # الطريقة 3: عبور نقطة المركز
        # ============================================
        if not crossed and len(self.positions) >= 4:
            cx, cy = self.get_center()
            prev_cx, prev_cy = self.positions[-2]
            
            if (prev_cx < zone_x1 or prev_cx > zone_x2 or 
                prev_cy < zone_y1 or prev_cy > zone_y2):
                if (zone_x1 <= cx <= zone_x2 and zone_y1 <= cy <= zone_y2):
                    crossed = True
                    direction = "IN"
                    confidence = 0.85
            
            elif (zone_x1 <= prev_cx <= zone_x2 and zone_y1 <= prev_cy <= zone_y2):
                if (cx < zone_x1 or cx > zone_x2 or 
                    cy < zone_y1 or cy > zone_y2):
                    crossed = True
                    direction = "OUT"
                    confidence = 0.85
        
        return direction, crossed

class DeepSORT:
    def __init__(self, max_age=30, min_hits=2, iou_threshold=0.25):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.next_id = 0
        self.tracks = []
        self.frame_count = 0
    
    def iou(self, bbox1, bbox2):
        x1, y1, w1, h1 = bbox1
        x2, y2, w2, h2 = bbox2
        
        x1_tl, y1_tl = x1 - w1/2, y1 - h1/2
        x2_tl, y2_tl = x2 - w2/2, y2 - h2/2
        x1_br, y1_br = x1 + w1/2, y1 + h1/2
        x2_br, y2_br = x2 + w2/2, y2 + h2/2
        
        xi1 = max(x1_tl, x2_tl)
        yi1 = max(y1_tl, y2_tl)
        xi2 = min(x1_br, x2_br)
        yi2 = min(y1_br, y2_br)
        
        if xi2 <= xi1 or yi2 <= yi1:
            return 0.0
        
        inter_area = (xi2 - xi1) * (yi2 - yi1)
        bbox1_area = w1 * h1
        bbox2_area = w2 * h2
        union_area = bbox1_area + bbox2_area - inter_area
        
        return inter_area / union_area if union_area > 0 else 0.0
    
    def associate_detections(self, detections):
        if len(self.tracks) == 0 or len(detections) == 0:
            return [], [], list(range(len(detections)))
        
        cost_matrix = np.zeros((len(self.tracks), len(detections)))
        for i, track in enumerate(self.tracks):
            track_state = track.get_state()
            for j, det in enumerate(detections):
                iou = self.iou(track_state, det)
                cost_matrix[i, j] = 1 - iou
        
        row_indices, col_indices = linear_sum_assignment(cost_matrix)
        
        matches = []
        unmatched_tracks = []
        unmatched_detections = []
        
        for i in range(len(row_indices)):
            track_idx = row_indices[i]
            det_idx = col_indices[i]
            if cost_matrix[track_idx, det_idx] < (1 - self.iou_threshold):
                matches.append((track_idx, det_idx))
            else:
                unmatched_tracks.append(track_idx)
        
        matched_tracks = [track_idx for track_idx, _ in matches]
        for i in range(len(self.tracks)):
            if i not in matched_tracks:
                if i not in unmatched_tracks:
                    unmatched_tracks.append(i)
        
        matched_detections = [det_idx for _, det_idx in matches]
        for j in range(len(detections)):
            if j not in matched_detections:
                if j not in unmatched_detections:
                    unmatched_detections.append(j)
        
        return matches, unmatched_tracks, unmatched_detections
    
    def update(self, detections):
        self.frame_count += 1
        
        for track in self.tracks:
            track.predict()
        
        matches, unmatched_tracks, unmatched_detections = self.associate_detections(detections)
        
        for track_idx, det_idx in matches:
            self.tracks[track_idx].update(detections[det_idx])
        
        new_tracks = []
        for i, track in enumerate(self.tracks):
            if i in unmatched_tracks:
                track.no_losses += 1
                if track.no_losses > self.max_age:
                    continue
            new_tracks.append(track)
        
        for det_idx in unmatched_detections:
            new_track = Track(detections[det_idx], self.next_id)
            self.next_id += 1
            new_tracks.append(new_track)
        
        self.tracks = new_tracks
    
    def get_tracks(self):
        return [track for track in self.tracks if track.hits >= self.min_hits]

# ============================================================
# 5. STORE COUNTER - ENHANCED ZONE
# ============================================================

class StoreCounter:
    def __init__(self):
        print("🔄 Loading YOLO model...")
        self.model = YOLO("yolov8n.pt")
        print("✅ YOLO model loaded")
        
        self.tracker = DeepSORT(max_age=30, min_hits=2, iou_threshold=0.25)
        
        # Store counts
        self.total_in = 0
        self.total_out = 0
        self.current_inside = 0
        
        # Track counted people
        self.counted_people = {}
        
        # Excel Logger
        self.excel_logger = ExcelLogger("E:\PPE-Detect-system\PPE-Detect-system\store_log.xlsx")
        
        # ============================================
        # ZONE CONFIGURATION - MEDIUM SIZE (for door)
        # ============================================
        self.zone_x1 = 150
        self.zone_y1 = 170
        self.zone_x2 = 450
        self.zone_y2 = 260
        
        # ============================================
        # OBJECT SIZE FILTER - تجاهل الأجسام الصغيرة جداً
        # ============================================
        self.min_object_width = 30
        self.min_object_height = 30
        self.ignore_small_objects = True
        
        # Cooldown
        self.crossing_cooldown = {}
        self.cooldown_seconds = 1.5
        
        self.last_sound_time = 0
        self.fps = 0
        self.frame_count = 0
        self.fps_time = time.time()
        
        self.track_colors = {}
        self.peak_inside = 0
        self.exit_button_rect = None
        self.running = True
        
        # Statistics
        self.total_detections = 0
        self.valid_counts = 0
        self.small_objects_ignored = 0
        
        print(f"📍 Zone: X1={self.zone_x1}, Y1={self.zone_y1}, X2={self.zone_x2}, Y2={self.zone_y2}")
        print(f"📍 Zone Size: {self.zone_x2 - self.zone_x1} x {self.zone_y2 - self.zone_y1} pixels")
        print(f"📍 Min Object Size: {self.min_object_width}x{self.min_object_height} pixels")
        print("✅ System ready - ENHANCED ZONE COUNTING")
    
    def get_color(self, track_id):
        if track_id not in self.track_colors:
            np.random.seed(track_id)
            color = tuple(np.random.randint(0, 255, 3).tolist())
            self.track_colors[track_id] = color
        return self.track_colors[track_id]
    
    def is_valid_object(self, track):
        """التحقق من أن الجسم ليس صغيراً جداً (تجاهل الذباب والحشرات)"""
        if not self.ignore_small_objects:
            return True
        width, height = track.get_size()
        if width < self.min_object_width or height < self.min_object_height:
            return False
        return True
    
    def process_frame(self, frame):
        height, width = frame.shape[:2]
        self.frame_count += 1
        
        if time.time() - self.fps_time >= 1.0:
            self.fps = self.frame_count
            self.frame_count = 0
            self.fps_time = time.time()
        
        # ============================================
        # DETECTION
        # ============================================
        results = self.model(frame, classes=[0], conf=0.25, verbose=False)
        
        detections = []
        if results and len(results[0].boxes) > 0:
            self.total_detections += len(results[0].boxes)
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                w = x2 - x1
                h = y2 - y1
                detections.append([float(cx), float(cy), float(w * 1.05), float(h * 1.05)])
        
        # ============================================
        # TRACKING
        # ============================================
        self.tracker.update(detections)
        tracks = self.tracker.get_tracks()
        
        # ============================================
        # COUNT - ZONE INTERSECTION
        # ============================================
        current_time = time.time()
        
        for track in tracks:
            track_id = track.track_id
            
            # Skip if already counted
            if track_id in self.counted_people:
                continue
            
            # ============================================
            # SKIP SMALL OBJECTS (ذباب، حشرات، أجسام صغيرة)
            # ============================================
            if not self.is_valid_object(track):
                self.counted_people[track_id] = "SKIPPED_SMALL"
                self.small_objects_ignored += 1
                continue
            
            # Check cooldown
            if track_id in self.crossing_cooldown:
                if current_time - self.crossing_cooldown[track_id] < self.cooldown_seconds:
                    continue
            
            # Check if object intersects with zone
            direction, crossed = track.check_zone_crossing(
                self.zone_x1, self.zone_y1, self.zone_x2, self.zone_y2
            )
            
            if crossed and direction:
                if direction == "IN":
                    self.total_in += 1
                    self.current_inside += 1
                    
                    if self.current_inside > self.peak_inside:
                        self.peak_inside = self.current_inside
                    
                    print(f"🚶 Object {track_id} ENTERED (IN: {self.total_in}, Inside: {self.current_inside})")
                else:
                    self.total_out += 1
                    self.current_inside -= 1
                    
                    print(f"🚶 Object {track_id} EXITED (OUT: {self.total_out}, Inside: {self.current_inside})")
                
                # Log to Excel
                if EXCEL_AVAILABLE:
                    self.excel_logger.add_entry(
                        entered_count=self.total_in,
                        exited_count=self.total_out,
                        inside_count=self.current_inside
                    )
                
                self.counted_people[track_id] = direction
                self.crossing_cooldown[track_id] = current_time
                self.valid_counts += 1
                
                if current_time - self.last_sound_time > 0.2:
                    play_sound()
                    self.last_sound_time = current_time
        
        # Clean up old tracks
        active_ids = [t.track_id for t in tracks]
        for tid in list(self.counted_people.keys()):
            if tid not in active_ids:
                del self.counted_people[tid]
        
        # ============================================
        # VISUALIZATION
        # ============================================
        
        # ============================================
        # DRAW ZONE
        # ============================================
        
        # Zone background (شفاف)
        overlay = frame.copy()
        cv2.rectangle(overlay, 
                     (self.zone_x1, self.zone_y1), 
                     (self.zone_x2, self.zone_y2), 
                     (0, 255, 255), -1)
        cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)
        
        # Zone border (سميك)
        cv2.rectangle(frame, 
                     (self.zone_x1, self.zone_y1), 
                     (self.zone_x2, self.zone_y2), 
                     (0, 255, 255), 3)
        
        # Zone glow effect
        for i in range(1, 3):
            cv2.rectangle(frame, 
                         (self.zone_x1 - i, self.zone_y1 - i), 
                         (self.zone_x2 + i, self.zone_y2 + i), 
                         (0, 255, 255), 1)
        
        # Zone label
        cv2.putText(frame, "🟡 COUNTING ZONE", (self.zone_x1 + 10, self.zone_y1 - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
        
        # Zone size info
        zone_width = self.zone_x2 - self.zone_x1
        zone_height = self.zone_y2 - self.zone_y1
        cv2.putText(frame, f"Size: {zone_width}x{zone_height}px", 
                    (self.zone_x1 + 10, self.zone_y1 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        
        # Direction indicators
        cv2.putText(frame, "⬇️ ENTER", (self.zone_x1 - 60, self.zone_y1 - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(frame, "⬆️ EXIT", (self.zone_x1 - 60, self.zone_y2 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        
        # ============================================
        # DRAW TRACKED OBJECTS
        # ============================================
        for track in tracks:
            track_id = track.track_id
            state = track.get_state()
            x, y, w, h = state
            
            color = self.get_color(track_id)
            
            x1 = int(x - w/2)
            y1 = int(y - h/2)
            x2 = int(x + w/2)
            y2 = int(y + h/2)
            
            # التحقق من كون الجسم صغيراً
            is_small = not self.is_valid_object(track)
            
            if is_small:
                # عرض الأجسام الصغيرة بشكل مختلف
                cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 128, 128), 1)
                cv2.putText(frame, "🔴 SMALL", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.3, (128, 128, 128), 1)
            else:
                # التحقق من وجود الجسم داخل المنطقة
                is_in_zone = track.check_zone_intersection(
                    self.zone_x1, self.zone_y1, self.zone_x2, self.zone_y2
                )
                
                # إبراز الجسم إذا كان داخل المنطقة
                if is_in_zone:
                    for i in range(1, 3):
                        cv2.rectangle(frame, (x1-i, y1-i), (x2+i, y2+i), (0, 255, 255), 1)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 3)
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                if track_id in self.counted_people:
                    dir = self.counted_people[track_id]
                    if dir == "IN":
                        cv2.rectangle(frame, (x1-2, y1-2), (x2+2, y2+2), (0, 255, 0), 3)
                        label = "✅ IN"
                    elif dir == "OUT":
                        cv2.rectangle(frame, (x1-2, y1-2), (x2+2, y2+2), (0, 0, 255), 3)
                        label = "✅ OUT"
                    else:
                        label = "⏭️"
                else:
                    if is_in_zone:
                        label = "🟡 IN ZONE"
                    else:
                        label = "⬆️" if y < self.zone_y1 else "⬇️"
                
                conf_text = f"{track.direction_confidence:.2f}" if track.direction_confidence > 0 else ""
                label = f"ID:{track_id} {label} {conf_text}"
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
                cv2.circle(frame, (int(x), int(y)), 3, (0, 255, 0), -1)
        
        # ============================================
        # STATS PANEL
        # ============================================
        panel_x = width - 250
        panel_y = 10
        panel_w = 230
        panel_h = 230
        
        cv2.rectangle(frame, (panel_x, panel_y), (panel_x + panel_w, panel_y + panel_h), (0, 0, 0), -1)
        cv2.rectangle(frame, (panel_x, panel_y), (panel_x + panel_w, panel_y + panel_h), (255, 255, 255), 1)
        
        cv2.putText(frame, "📊 STORE STATS", (panel_x + 10, panel_y + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        cv2.putText(frame, f"IN: {self.total_in}", (panel_x + 10, panel_y + 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
        cv2.putText(frame, f"OUT: {self.total_out}", (panel_x + 10, panel_y + 72),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
        cv2.putText(frame, f"INSIDE: {self.current_inside}", (panel_x + 10, panel_y + 94),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
        cv2.putText(frame, f"PEAK: {self.peak_inside}", (panel_x + 10, panel_y + 116),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 165, 0), 2)
        cv2.putText(frame, f"TOTAL: {self.total_in + self.total_out}", (panel_x + 10, panel_y + 138),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        
        if self.total_detections > 0:
            acc = (self.valid_counts / max(1, self.total_detections)) * 100
            acc_text = f"ACCURACY: {acc:.1f}%"
            color_acc = (0, 255, 0) if acc > 90 else (0, 255, 255) if acc > 70 else (0, 0, 255)
            cv2.putText(frame, acc_text, (panel_x + 10, panel_y + 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_acc, 2)
        
        # معلومات عن الأجسام الصغيرة
        cv2.putText(frame, f"🔴 Small Objects: {self.small_objects_ignored}", (panel_x + 10, panel_y + 182),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 200, 255), 1)
        
        small_status = "🚫 SMALL" if self.ignore_small_objects else "🔴 ALLOW"
        cv2.putText(frame, small_status, (panel_x + 10, panel_y + 202),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 200, 255), 1)
        
        # FPS
        cv2.putText(frame, f"FPS: {self.fps}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # Status
        cv2.putText(frame, "🔊" if SOUND_ENABLED else "🔇", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
        cv2.putText(frame, "📊" if EXCEL_AVAILABLE else "❌", (10, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0) if EXCEL_AVAILABLE else (0, 0, 255), 2)
        
        # ============================================
        # EXIT BUTTON
        # ============================================
        btn_x = width - 120
        btn_y = height - 50
        btn_w = 100
        btn_h = 35
        
        cv2.rectangle(frame, (btn_x, btn_y), (btn_x + btn_w, btn_y + btn_h), (0, 0, 200), -1)
        cv2.rectangle(frame, (btn_x, btn_y), (btn_x + btn_w, btn_y + btn_h), (255, 255, 255), 2)
        cv2.putText(frame, "✖ EXIT", (btn_x + 18, btn_y + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        self.exit_button_rect = (btn_x, btn_y, btn_x + btn_w, btn_y + btn_h)
        
        # Controls
        cv2.putText(frame, "q:Quit | r:Reset | e:Report | Click EXIT", (10, height - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        
        return frame
    
    def show_excel_report(self):
        if not EXCEL_AVAILABLE:
            print("⚠️ Excel not available")
            return
        
        print("\n" + "=" * 70)
        print("📊 EXCEL REPORT")
        print("=" * 70)
        print(f"📡 Camera IP: {CAMERA_IP}")
        print(f"📁 File: {self.excel_logger.filename}")
        
        try:
            df = pd.read_excel(self.excel_logger.filename, engine='openpyxl')
            print(f"\n📋 Total Records: {len(df)}")
            print("\n📊 Last 5 Records:")
            print(df.tail(5).to_string(index=False))
        except:
            pass
        
        print("\n" + "=" * 70)
        print(f"🔴 Small Objects Ignored: {self.small_objects_ignored}")
        print("=" * 70)

# ============================================================
# 6. CAMERA THREAD
# ============================================================

class CameraThread(threading.Thread):
    def __init__(self, camera_url):
        super().__init__(daemon=True)
        self.camera_url = camera_url
        self.running = True
        self.frame = None
        self.lock = threading.Lock()

    def get_frame(self):
        with self.lock:
            return self.frame.copy() if self.frame is not None else None

    def run(self):
        cap = None
        while self.running:
            try:
                if cap is None or not cap.isOpened():
                    print("📷 Connecting to camera...")
                    cap = cv2.VideoCapture(self.camera_url)
                    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
                    cap.set(cv2.CAP_PROP_FPS, 30)
                    if not cap.isOpened():
                        time.sleep(1)
                        continue
                    print("✅ Camera connected")
                
                ret, frame = cap.read()
                if not ret or frame is None:
                    time.sleep(0.001)
                    continue
                
                with self.lock:
                    self.frame = frame
                    
            except Exception as e:
                if cap:
                    cap.release()
                    cap = None
                time.sleep(1)
        
        if cap:
            cap.release()

# ============================================================
# 7. MAIN
# ============================================================

def main():
    IP = "192.168.1.51"
    USERNAME = "admin"
    PASSWORD = "Manager10g_%23"
    
    STREAM_URLS = [
        f"rtsp://{USERNAME}:{PASSWORD}@{IP}:554/Streaming/Channels/102",
        f"rtsp://{USERNAME}:{PASSWORD}@{IP}:554/Streaming/Channels/101",
    ]
    CAMERA_URL = STREAM_URLS[0]
    
    DISPLAY_WIDTH = 900
    DISPLAY_HEIGHT = 650
    WINDOW_NAME = "🏪 Store Counter - Enhanced Zone"
    
    print("\n" + "=" * 70)
    print("🏪 STORE PEOPLE COUNTER - ENHANCED ZONE")
    print("   ✅ Zone covers the door area")
    print("   ✅ Any object (body part or clothing) touching zone = counted")
    print("   ✅ Small objects (flies, insects) are ignored")
    print("   ✅ Each person counted ONCE")
    print("   ✅ Click EXIT button to quit")
    print("=" * 70)
    print(f"📷 Camera: {IP}")
    print(f"📡 Local IP: {CAMERA_IP}")
    print("=" * 70)
    print("\nControls:")
    print("  'q' - Quit")
    print("  'r' - Reset all counters")
    print("  'e' - Show Excel report")
    print("  Click 'EXIT' button to quit")
    print("=" * 70 + "\n")
    
    counter = StoreCounter()
    camera = CameraThread(CAMERA_URL)
    camera.start()
    
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WINDOW_NAME, DISPLAY_WIDTH, DISPLAY_HEIGHT)
    
    exit_clicked = False
    
    def mouse_callback(event, x, y, flags, param):
        nonlocal exit_clicked
        if event == cv2.EVENT_LBUTTONDOWN:
            if counter.exit_button_rect:
                btn_x, btn_y, btn_x2, btn_y2 = counter.exit_button_rect
                if btn_x <= x <= btn_x2 and btn_y <= y <= btn_y2:
                    print("🛑 Exit button clicked - Closing application...")
                    exit_clicked = True
                    cv2.destroyAllWindows()
                    os._exit(0)
    
    cv2.setMouseCallback(WINDOW_NAME, mouse_callback)
    
    try:
        while True:
            if exit_clicked:
                break
            
            frame = camera.get_frame()
            if frame is None:
                time.sleep(0.001)
                continue
            
            processed = counter.process_frame(frame)
            cv2.imshow(WINDOW_NAME, processed)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q') or key == 27:
                print("🛑 'q' pressed - Closing application...")
                break
            elif key == ord('r'):
                counter.total_in = 0
                counter.total_out = 0
                counter.current_inside = 0
                counter.peak_inside = 0
                counter.counted_people.clear()
                counter.crossing_cooldown.clear()
                counter.total_detections = 0
                counter.valid_counts = 0
                counter.small_objects_ignored = 0
                counter.tracker = DeepSORT(max_age=30, min_hits=2, iou_threshold=0.25)
                print("🔄 All counters reset!")
            elif key == ord('e'):
                counter.show_excel_report()
            
    except KeyboardInterrupt:
        print("\n👋 Interrupted")
    finally:
        if EXCEL_AVAILABLE:
            counter.excel_logger.save_final()
        
        camera.running = False
        camera.join(1)
        cv2.destroyAllWindows()
        
        print("\n" + "=" * 70)
        print("📊 FINAL STORE REPORT")
        print("=" * 70)
        print(f"   🚶 Objects Entered: {counter.total_in}")
        print(f"   🚶 Objects Exited: {counter.total_out}")
        print(f"   🟡 Currently Inside: {counter.current_inside}")
        print(f"   🔺 Peak Inside: {counter.peak_inside}")
        print(f"   📊 Total Objects: {counter.total_in + counter.total_out}")
        print(f"   🔴 Small Objects Ignored: {counter.small_objects_ignored}")
        if counter.total_detections > 0:
            acc = (counter.valid_counts / counter.total_detections) * 100
            print(f"   🎯 Accuracy: {acc:.1f}%")
        print(f"   📡 Camera IP: {CAMERA_IP}")
        print("=" * 70)
        print("👋 Program terminated")

if __name__ == "__main__":
    main()

📡 Camera IP: 192.168.1.128
📊 Excel logging enabled

🏪 STORE PEOPLE COUNTER - ENHANCED ZONE
   ✅ Zone covers the door area
   ✅ Any object (body part or clothing) touching zone = counted
   ✅ Small objects (flies, insects) are ignored
   ✅ Each person counted ONCE
   ✅ Click EXIT button to quit
📷 Camera: 192.168.1.51
📡 Local IP: 192.168.1.128

Controls:
  'q' - Quit
  'r' - Reset all counters
  'e' - Show Excel report
  Click 'EXIT' button to quit

🔄 Loading YOLO model...
✅ YOLO model loaded
📍 Zone: X1=150, Y1=170, X2=450, Y2=260
📍 Zone Size: 300 x 90 pixels
📍 Min Object Size: 30x30 pixels
✅ System ready - ENHANCED ZONE COUNTING
📷 Connecting to camera...
✅ Camera connected

👋 Interrupted
📊 Final data saved to: E:\PPE-Detect-system\PPE-Detect-system\store_log.xlsx

📊 FINAL STORE REPORT
   🚶 Objects Entered: 0
   🚶 Objects Exited: 0
   🟡 Currently Inside: 0
   🔺 Peak Inside: 0
   📊 Total Objects: 0
   🔴 Small Objects Ignored: 0
   📡 Camera IP: 192.168.1.128
👋 Program terminated
